In [1]:
import tkinter as tk
from tkinter import ttk, messagebox
import mysql.connector
print("MySQL connected module working!")

MySQL connected module working!


In [2]:
mydb = mysql.connector.connect(
    host="localhost",
    user="root",
    password=""
)
cursor = mydb.cursor()
cursor.execute("drop database if exists lippan_db")
cursor.execute("create database lippan_db")

In [ ]:
mydb = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database="lippan_db"
)

cursor = mydb.cursor()
cursor.execute("""
create table if not exists customers (
    id int auto_increment PRIMARY KEY,
    name VARCHAR(100),
    phone VARCHAR(15)
);
""")

cursor.execute("""
create table if not exists orders (
    id int auto_increment PRIMARY KEY,
    customer_id int,
    design varchar(100),
    quantity int,
    price float,
    discount float,
    status varchar(20),
    total float,
    order_date DATETIME DEFAULT CURRENT_TIMESTAMP,
    foreign key (customer_id) REFERENCES customers(id)
);
""")

In [ ]:
import tkinter as tk
from tkinter import ttk
import mysql.connector
import matplotlib.pyplot as plt

mydb = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database="lippan_db"
)
cursor = mydb.cursor()

root = tk.Tk()
root.title("Lippan Art Order Management System")
root.geometry("1100x600")
root.config(bg="#fff")

oid = tk.StringVar()
name = tk.StringVar()
phone = tk.StringVar()
design = tk.StringVar()
price = tk.StringVar()
quantity = tk.StringVar()
discount = tk.StringVar()
status = tk.StringVar(value="Pending")
search_name = tk.StringVar()

def get_customer_id():
    cursor.execute("select id from customers where phone=%s", (phone.get(),))
    result = cursor.fetchone()

    if result:
        return result[0]
    else:
        cursor.execute(
            "insert into customers (name, phone) values (%s,%s)",
            (name.get(), phone.get())
        )
        mydb.commit()
        return cursor.lastrowid

def calculate_total():
    subtotal = float(price.get()) * int(quantity.get())
    discount_amount = (subtotal * float(discount.get())) / 100
    total = subtotal - discount_amount
    return subtotal, discount_amount, total


    
def add_order():
    try:
        cid = get_customer_id()
        _, _, total = calculate_total()

        cursor.execute("""
        insert into orders (customer_id, design, quantity, price, discount, status, total)
        VALUES (%s,%s,%s,%s,%s,%s,%s)
        """, (cid, design.get(), quantity.get(), price.get(), discount.get(), status.get(), total))

        mydb.commit()
        fetch_data()
        clear_fields()

    except:
        print("Error adding order")
        
def update_order():
    if oid.get() == "":
        print("Select a record first")
        return

    _, _, total = calculate_total()
    
    cursor.execute("""
    update orders 
    set design=%s, quantity=%s, price=%s, discount=%s, status=%s, total=%s
    where id=%s
    """, (
        design.get(),
        int(quantity.get()),
        float(price.get()),
        float(discount.get()),
        status.get(),
        total,
        int(oid.get())
    ))

    mydb.commit()
    fetch_data()
    clear_fields()


def delete_order():
    if oid.get():
        cursor.execute("DELETE FROM orders WHERE id=%s", (oid.get(),))
        mydb.commit()
        fetch_data()


def clear_fields():
    oid.set("")
    name.set("")
    phone.set("")
    design.set("")
    price.set("")
    quantity.set("")
    discount.set("")
    status.set("Pending")


def fetch_data():
    for row in tree.get_children():
        tree.delete(row)

    cursor.execute("""
    select orders.id, customers.name, customers.phone,
    orders.design, orders.quantity, orders.price,
    orders.discount, orders.status, orders.total , orders.order_date
    from orders
    join customers on orders.customer_id = customers.id
    """)

    for row in cursor.fetchall():
        if row[7] == "Completed":
            tree.insert("", tk.END, values=row, tags=("done",))
        else:
            tree.insert("", tk.END, values=row)


def on_select(event):
    data = tree.item(tree.focus(), "values")
    if data:
        oid.set(data[0])
        name.set(data[1])
        phone.set(data[2])
        design.set(data[3])
        quantity.set(data[4])
        price.set(data[5])
        discount.set(data[6])
        status.set(data[7])


def filter_status(s):
    for row in tree.get_children():
        tree.delete(row)

    cursor.execute("""
    select orders.id, customers.name, customers.phone,
    orders.design, orders.quantity, orders.price,
    orders.discount, orders.status, orders.total
    from orders
    join customers on orders.customer_id = customers.id
    where orders.status=%s
    """, (s,))

    for row in cursor.fetchall():
        tree.insert("", tk.END, values=row)


def sort_total():
    for row in tree.get_children():
        tree.delete(row)

    cursor.execute("""
    select orders.id, customers.name, customers.phone,
    orders.design, orders.quantity, orders.price,
    orders.discount, orders.status, orders.total
    from orders
    join customers ON orders.customer_id = customers.id
    order by orders.total desc
    """)

    for row in cursor.fetchall():
        tree.insert("", tk.END, values=row)

    
def search_by_name():
    for row in tree.get_children():
        tree.delete(row)

    cursor.execute("""
    select orders.id, customers.name, customers.phone,
    orders.design, orders.quantity, orders.price,
    orders.discount, orders.status, orders.total
    from orders
    join customers on orders.customer_id = customers.id
    where customers.name like %s
    """, ("%" + search_name.get() + "%",))

    for row in cursor.fetchall():
        tree.insert("", tk.END, values=row)

def show_pie_chart():
    cursor.execute("SELECT status, COUNT(*) FROM orders GROUP BY status")
    data = cursor.fetchall()

    labels = [row[0] for row in data]
    values = [row[1] for row in data]

    colors = ["#ff9999", "#66b3ff"]

    plt.pie(values, labels=labels, autopct='%1.1f%%', colors=colors)
    plt.legend(title="Order Status")
    plt.title("Order Distribution")
    plt.show()

def sales_graph():
    cursor.execute("SELECT status, SUM(total) FROM orders GROUP BY status")
    data = cursor.fetchall()

    labels = [row[0] for row in data]
    values = [row[1] for row in data]

    colors = ["#ff7f7f", "#7fff7f"]  # red & green

    plt.bar(labels, values, color=colors)

    # Add value labels on top
    for i, v in enumerate(values):
        plt.text(i, v, str(round(v, 2)), ha='center', va='bottom')

    plt.title("Total Sales by Status")
    plt.xlabel("Order Status")
    plt.ylabel("Sales Amount")
    plt.legend(labels)
    plt.grid(axis='y', linestyle='--', alpha=0.5)

    plt.show()

    
def generate_bill():
    selected = tree.focus()
    if not selected:
        return

    data = tree.item(selected, "values")
    if not data:
        return

    oid_val, cname, cphone, design_val, qty, price_val, disc, stat, total_val , date_val = data

    subtotal = float(price_val) * int(qty)
    discount_amount = (subtotal * float(disc)) / 100

    bill_text.delete("1.0", tk.END)

    bill = f"""
==============================
      LIPPAN ART SHOP
==============================
Order ID   : {oid_val}
Customer   : {cname}
Phone      : {cphone}

Design     : {design_val}
Price      : ₹{price_val}
Quantity   : {qty}
Date       : {date_val}

------------------------------
Subtotal   : ₹{subtotal}
Discount   : {disc}% (-₹{round(discount_amount,2)})
------------------------------
TOTAL      : ₹{total_val}
Status     : {stat}
==============================

   Thank you for your order 
"""
    bill_text.insert(tk.END, bill)

def total_sales():
    cursor.execute("SELECT SUM(total) FROM orders")
    result = cursor.fetchone()[0]

    bill_text.delete("1.0", tk.END)
    bill_text.insert(tk.END, f"""
==============================
      TOTAL SALES
==============================
₹ {result if result else 0}
==============================
""")
        
tk.Label(root, text="Lippan Art Order Management System",
         font=("Poppins", 18, "bold"),
         bg="#fff").pack(pady=10)
main = tk.Frame(root, bg="#f5e6d3")
main.pack(fill="both", expand=True, padx=10, pady=10)

left = tk.Frame(main, bg="#f5e6d3")
left.pack(side="left", fill="y", padx=10)

tk.Label(left, text="Search Name", bg="#f5e6d3").grid(row=0, column=0, pady=8, padx=5, sticky="w")
tk.Entry(left, textvariable=search_name).grid(row=0, column=1, pady=8)
tk.Button(left, text="Search", width=26, command=search_by_name, bg="#d9a066").grid(row=1, column=0, columnspan=2, pady=10)

tk.Label(left, text="Name", bg="#f5e6d3").grid(row=2, column=0, pady=8, padx=5, sticky="w")
tk.Entry(left, textvariable=name, width=20).grid(row=2, column=1, pady=8)

tk.Label(left, text="Phone", bg="#f5e6d3").grid(row=3, column=0, pady=8, padx=5, sticky="w")
tk.Entry(left, textvariable=phone, width=20).grid(row=3, column=1, pady=8)

tk.Label(left, text="Design", bg="#f5e6d3").grid(row=4, column=0, pady=8, padx=5, sticky="w")
tk.Entry(left, textvariable=design, width=20).grid(row=4, column=1, pady=8)

tk.Label(left, text="Price", bg="#f5e6d3").grid(row=5, column=0, pady=8, padx=5, sticky="w")
tk.Entry(left, textvariable=price, width=20).grid(row=5, column=1, pady=8)

tk.Label(left, text="Quantity", bg="#f5e6d3").grid(row=6, column=0, pady=8, padx=5, sticky="w")
tk.Entry(left, textvariable=quantity, width=20).grid(row=6, column=1, pady=8)

tk.Label(left, text="Discount (%)", bg="#f5e6d3").grid(row=7, column=0, pady=8, padx=5, sticky="w")
tk.Entry(left, textvariable=discount, width=20).grid(row=7, column=1, pady=8)

tk.Button(left, text="Total Sales", width=26,
          command=total_sales, bg="#d9a066").grid(row=17, column=0, columnspan=2, pady=5)

tk.Label(left, text="Status", bg="#f5e6d3").grid(row=8, column=0, pady=8, padx=5, sticky="w")
ttk.Combobox(left, textvariable=status,
             values=["Pending", "Completed"],
             state="readonly",
             width=18).grid(row=8, column=1, pady=8)

btn_width = 12


tk.Button(left, text="Add", width=btn_width, command=add_order, bg="#d9a066").grid(row=9, column=0, padx=5, pady=5)
tk.Button(left, text="Update", width=btn_width, command=update_order, bg="#d9a066").grid(row=9, column=1, padx=5, pady=5)

tk.Button(left, text="Delete", width=btn_width, command=delete_order, bg="#d9a066").grid(row=10, column=0, padx=5, pady=5)
tk.Button(left, text="Clear", width=btn_width, command=clear_fields, bg="#d9a066").grid(row=10, column=1, padx=5, pady=5)

tk.Button(left, text="Pending", width=btn_width, command=lambda: filter_status("Pending"), bg="#d9a066").grid(row=11, column=0, padx=5, pady=5)
tk.Button(left, text="Completed", width=btn_width, command=lambda: filter_status("Completed"), bg="#d9a066").grid(row=11, column=1, padx=5, pady=5)

tk.Button(left, text="Show All", width=btn_width, command=fetch_data, bg="#d9a066").grid(row=12, column=0, padx=5, pady=5)
tk.Button(left, text="Sort Total", width=btn_width, command=sort_total, bg="#d9a066").grid(row=12, column=1, padx=5, pady=5)

tk.Button(left, text="Graph", width=26, command=show_pie_chart, bg="#d9a066").grid(row=13, column=0, columnspan=2, pady=10)
tk.Button(left, text="Sales Graph", command=sales_graph, bg="#d9a066").grid(row=14, column=0, columnspan=2, pady=5)


tk.Button(left, text="Generate Bill", width=26,
          command=generate_bill, bg="#d9a066").grid(row=15, column=0, columnspan=2, pady=10)


right = tk.Frame(main)
right.pack(side="left", fill="both", expand=True)

cols = ("ID","Name","Phone","Design","Quantity","Price","Discount","Status","Total","Date")
tree = ttk.Treeview(right, columns=cols, show="headings")
tree.tag_configure("done", background="#d4edda")

tree.heading("ID", text="ID")
tree.column("ID", width=40, anchor="center")

tree.heading("Name", text="Name")
tree.column("Name", width=120)

tree.heading("Phone", text="Phone")
tree.column("Phone", width=100)

tree.heading("Design", text="Design")
tree.column("Design", width=120)

tree.heading("Quantity", text="Quantity")
tree.column("Quantity", width=60, anchor="center")

tree.heading("Price", text="Price")
tree.column("Price", width=80, anchor="center")

tree.heading("Discount", text="Discount")
tree.column("Discount", width=80, anchor="center")

tree.heading("Status", text="Status")
tree.column("Status", width=90, anchor="center")

tree.heading("Total", text="Total")
tree.column("Total", width=100, anchor="center")
tree.heading("Date", text="Date")
tree.column("Date", width=140)
bill_frame = tk.Frame(right, bg="white", bd=2, relief="solid")
bill_frame.pack(fill="x", padx=5, pady=5)

bill_text = tk.Text(bill_frame, height=15, font=("Courier", 10))
bill_text.pack(fill="both", expand=True)

right.config(width=700)   
left.config(width=300)    
tree.pack(fill="both", expand=True)

tree.bind("<<TreeviewSelect>>", on_select)

fetch_data()

root.mainloop()

In [6]:

!pip install matplotlib

In [9]:
pip install matplotlib

  Using cached matplotlib-3.10.8-cp313-cp313-win_amd64.whl.metadata (52 kB)
  Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.62.1-cp313-cp313-win_amd64.whl.metadata (119 kB)
  Using cached kiwisolver-1.5.0-cp313-cp313-win_amd64.whl.metadata (5.2 kB)
Using cached matplotlib-3.10.8-cp313-cp313-win_amd64.whl (8.1 MB)
Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl (226 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.62.1-cp313-cp313-win_amd64.whl (2.3 MB)
Using cached kiwisolver-1.5.0-cp313-cp313-win_amd64.whl (73 kB)

   -------- ------------------------------- 1/5 [fonttools]
   -------- ------------------------------- 1/5 [fonttools]
   -------- ------------------------------- 1/5 [fonttools]
   -------- ------------------------------- 1/5 [fonttools]
   -------- ------------------------------- 1/5 [fonttools]
   -------- --------

In [1]:
import matplotlib.pyplot as plt
print("Matplotlib installed successfully!")

Matplotlib installed successfully!
